# S4.4 - Join strategies
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** The same join can run as broadcast, shuffle hash or sort-merge; the plan shows which

In [0]:
# ============================================================
# S4.4 - Cell 2
# Join Strategies:
# Broadcast Hash, Shuffle Hash, Sort-Merge
#
# Goal:
# 1. Request different join strategies with hints.
# 2. Read the join operator shown by explain().
# 3. Compare requested strategy with the observed plan.
# 4. Confirm strategy changes HOW Spark executes the join,
#    not WHAT rows the logical join returns.
# 5. Show that an explicit BROADCAST hint can change a join
#    that Spark previously planned as shuffle-based.
#
# Important:
# - A hint is a request, not an absolute guarantee.
# - explain() may show an AQE plan with isFinalPlan=false.
# - Query Profile is the stronger runtime evidence.
#
# Scope:
# - Broadcast Hash Join
# - Shuffle Hash Join
# - Sort-Merge Join
#
# Deferred:
# - duplicate/cardinality issues -> S4.5
# ============================================================

import io
import re
import contextlib

import pyspark.sql.functions as F


print("=== S4.4: JOIN STRATEGIES ===")
print()


# ============================================================
# Step 1 - Source Data
#
# Use the same relationship for every strategy:
#
# orders.o_custkey
#       =
# customer.c_custkey
#
# Keeping the logical join identical lets us compare
# execution strategies fairly.
# ============================================================

orders = (
    spark.table("samples.tpch.orders")
    .select(
        "o_orderkey",
        "o_custkey"
    )
)

customer = (
    spark.table("samples.tpch.customer")
    .select(
        "c_custkey",
        "c_nationkey"
    )
)


print("Orders DataFrame:")
orders.show(5)

print("customer DataFrame:")
customer.show(5)

In [0]:
# ============================================================
# Step 2 - Helper: Capture explain() Output
#
# Avoid _jdf internals because this notebook runs on
# Databricks Serverless / Spark Connect.
# ============================================================

def explain_text(df):

    buf = io.StringIO()

    with contextlib.redirect_stdout(buf):

        df.explain(
            mode="formatted"
        )

    return buf.getvalue()


In [0]:
# ============================================================
# Step 3 - Helper: Find Join Operator
#
# Handle both generic Spark and Photon names.
#
# Examples:
#
# BroadcastHashJoin
# PhotonBroadcastHashJoin
#
# ShuffledHashJoin
# PhotonShuffledHashJoin
#
# SortMergeJoin
# PhotonSortMergeJoin
# ============================================================

def join_op(df):

    plan = explain_text(df)

    patterns = [
        r"\bPhotonBroadcastHashJoin\b",
        r"\bBroadcastHashJoin\b",

        r"\bPhotonShuffledHashJoin\b",
        r"\bShuffledHashJoin\b",

        r"\bPhotonSortMergeJoin\b",
        r"\bSortMergeJoin\b"
    ]

    found = []

    for pattern in patterns:

        matches = re.findall(pattern,plan)

        for match in matches:

            if match not in found:
                found.append(match)

    return (
        found
        if found
        else ["(no join operator found)"]
    )

In [0]:
# ============================================================
# Step 4 - Build the Same Logical Join Using Different Hints
#
# Previous S4.3 observation:
#
# orders × customer
# → PhotonShuffledHashJoin
#
# Predictions / requests:
#
# no hint
# → optimizer decides
#
# MERGE
# → request SortMergeJoin
#
# SHUFFLE_HASH
# → request ShuffledHashJoin
#
# BROADCAST
# → request BroadcastHashJoin using customer as build side
#
# Important:
# Actual plan may differ because hints are not guaranteed.
# ============================================================

o1 = orders.alias("o")
c1 = customer.alias("c")

condition_no_hint = (F.col("o.o_custkey") == F.col("c.c_custkey"))

j_no_hint = (
    o1.join(c1, condition_no_hint,"inner"))

# ------------------------------------------------------------
# MERGE hint
# ------------------------------------------------------------

o2 = orders.alias("o")

c2 = (customer.hint("merge").alias("c"))

j_merge = (o2.join(c2,F.col("o.o_custkey") == F.col("c.c_custkey"),"inner"))

# ------------------------------------------------------------
# SHUFFLE_HASH hint
# ------------------------------------------------------------

o3 = orders.alias("o")

c3 = (customer.hint("shuffle_hash").alias("c"))

j_shuffle_hash = (o3.join(c3,F.col("o.o_custkey") == F.col("c.c_custkey"), "inner"))

# ------------------------------------------------------------
# BROADCAST hint
#
# We explicitly request customer as the broadcast side.
# ------------------------------------------------------------

o4 = orders.alias("o")

c4 = (customer.hint("broadcast").alias("c"))

j_broadcast = (o4.join(c4,F.col("o.o_custkey") == F.col("c.c_custkey"),"inner"))

plans = {
    "no hint": j_no_hint,
    "MERGE": j_merge,
    "SHUFFLE_HASH": j_shuffle_hash,
    "BROADCAST": j_broadcast}

requested = {
    "no hint": "optimizer decides",
    "MERGE": "SortMergeJoin",
    "SHUFFLE_HASH": "ShuffledHashJoin",
    "BROADCAST": "BroadcastHashJoin"}

In [0]:
# ============================================================
# Step 5 - Compare Requested vs Plan Operator
#
# These are explain-plan observations.
#
# If AQE shows:
#
# isFinalPlan=false
#
# the runtime plan may still adapt.
# ============================================================

print("=== STEP 5: JOIN OPERATOR PER REQUEST ===")

operators = {}

for name, df in plans.items():

    ops = join_op(df)

    operators[name] = ops

    actual = (
        ", ".join(ops)
    )

    print(
        f"RESULT {name:<15} "
        f"requested={requested[name]:<22} "
        f"plan={actual}"
    )


print()

In [0]:
# ============================================================
# Step 6 - Execute Every Strategy
#
# Strategy should affect physical execution only.
#
# The logical result should remain identical.
# ============================================================

print(
    "=== STEP 6: ROW COUNT PER STRATEGY ==="
)


counts = {}


for name, df in plans.items():

    n = df.count()

    counts[name] = n

    print(
        f"RESULT {name:<15} "
        f"rows={n:,}"
    )


all_equal = (
    len(
        set(
            counts.values()
        )
    )
    ==
    1
)


print(
    f"RESULT all row counts equal: "
    f"{all_equal}"
)

print()


In [0]:
# ============================================================
# Step 7 - Broadcast Hint Follow-up From S4.3
#
# In S4.3:
#
# orders × customer without hint
#
# was observed as:
#
# PhotonShuffledHashJoin
#
# Here we compare the SAME logical join with:
#
# customer.hint("broadcast")
#
# We do NOT attempt to turn the Serverless broadcast
# threshold off.
#
# Instead we ask:
#
# Can an explicit BROADCAST hint change the strategy that
# Spark selected without a hint?
# ============================================================

print(
    "=== STEP 7: NO HINT vs BROADCAST HINT ==="
)


print(
    "No-hint operator :",
    operators["no hint"]
)

print(
    "Broadcast operator:",
    operators["BROADCAST"]
)


broadcast_changed_plan = (
    operators["no hint"]
    !=
    operators["BROADCAST"]
)


print(
    "Plan operator changed:",
    broadcast_changed_plan
)

print(
    "Logical results equal:",
    counts["no hint"]
    ==
    counts["BROADCAST"]
)

print()

In [0]:
# ============================================================
# Step 8 - Strategy Meaning
# ============================================================

print("=== STEP 8: STRATEGY MEANING ===")

print("""
BROADCAST HASH JOIN

Small side
    ↓
copied to workers

Large side
    ↓
does not require the same large join-key redistribution

Best candidate:
large relation + safely small relation


SHUFFLE HASH JOIN

Both sides
    ↓
partitioned by join key
    ↓
matching partitions meet
    ↓
hash table built for one side

Requires network shuffle.


SORT-MERGE JOIN

Both sides
    ↓
shuffle by join key
    ↓
sort within partitions
    ↓
merge matching sorted keys

Requires shuffle and sorting.


IMPORTANT

These strategies change:

HOW Spark performs the join

They do not change:

WHAT rows the logical join should return.
""")

In [0]:
# ============================================================
# Step 9 - Hint Rules
#
# Spark supports strategy hints such as:
#
# BROADCAST
# MERGE
# SHUFFLE_HASH
#
# But a hint is not an unconditional command.
#
# Actual strategy can depend on:
#
# - join type
# - optimizer support
# - statistics
# - AQE
# - execution engine / Photon
# ============================================================

print("=== STEP 9: HINT RULE ===")

print("""
Hint requested
      ↓
Optimizer considers it
      ↓
Supported for this join?
      ↓
Yes → strategy may be used
No  → optimizer may choose something else

Therefore always inspect the plan.
""")

In [0]:
# ============================================================
# Step 10 - S4.4 Checkpoint
#
# Copy these ACTUAL observations into the final %md.
# ============================================================

print(
    "=== STEP 10: S4.4 CHECKPOINT ==="
)


for name in plans:

    print(
        f"{name:<15} "
        f"requested={requested[name]:<22} "
        f"plan={operators[name]} "
        f"rows={counts[name]:,}"
    )


print(
    f"All logical results equal : "
    f"{all_equal}"
)


print(
    f"Broadcast changed operator : "
    f"{broadcast_changed_plan}"
)


print()

In [0]:
# ============================================================
# Step 11 - Final Learning Summary
# ============================================================

print("=== FINAL LEARNING SUMMARY ===")


print("""
JOIN STRATEGY

Logical join:
same

Physical execution:
can be different


BROADCAST HASH

small side copied to workers
→ can avoid a large-side shuffle


SHUFFLE HASH

both sides partitioned by join key
→ hash-based matching


SORT-MERGE

both sides partitioned
+
sorted
→ matching sorted keys are merged


HINTS

broadcast
merge
shuffle_hash

→ request a physical strategy


IMPORTANT

Hint != guarantee

Always inspect the actual plan.


SERVERLESS NOTE

We do NOT rely on manually disabling
spark.sql.autoBroadcastJoinThreshold

because that configuration was not reliably
exposed in our Serverless workspace.


S4.3 FOLLOW-UP

Instead of trying to disable auto broadcast:

compare the same orders × customer join

no hint
vs
explicit BROADCAST hint

to observe whether the hint changes the
optimizer's chosen physical strategy.
""")

# S4.4 — Key Takeaways: Join Strategies

## 1. The Three Strategies

| Strategy | How it works | Main data movement |
|---|---|---|
| Broadcast hash join | small side is copied to workers and used to build a hash table | large side can avoid join-key shuffle |
| Shuffle hash join | both sides are shuffled by join key; a hash table is built per matching partition | both sides shuffled |
| Sort-merge join | both sides are shuffled by join key, sorted, then merged | both sides shuffled and sorted |

### Broadcast Hash Join

```text
Large relation
+
small relation
      ↓
broadcast small side
      ↓
hash lookup on workers
```

Useful when one relation is safely small.

### Shuffle Hash Join

```text
Both sides
      ↓
shuffle by join key
      ↓
matching partitions
      ↓
build hash table for one side
```

### Sort-Merge Join

```text
Both sides
      ↓
shuffle by join key
      ↓
sort
      ↓
merge matching keys
```

Pure non-equi joins such as:

```text
a.x > b.y
```

do not have equality keys for a standard hash join.

Cross joins also use different physical strategies.

Those operators were not part of our verified S4.4 checkpoint.

---

## 2. Join Strategy Hints

Spark supports strategy hints such as:

```text
BROADCAST
MERGE
SHUFFLE_HASH
SHUFFLE_REPLICATE_NL
```

Example:

```python
orders.join(
    customer.hint("merge"),
    cond
)
```

The hint requests a physical strategy.

It is not an unconditional command.

Therefore:

```text
hint requested
      ↓
optimizer evaluates it
      ↓
strategy supported?
      ↓
may use requested strategy
```

Always inspect the physical plan.

---

## 3. Logical Result vs Physical Strategy

The logical join remained the same:

```text
orders.o_custkey
=
customer.c_custkey
```

Only the requested physical strategy changed.

Observed row count for every version:

```text
7,500,000
```

Therefore:

```text
Broadcast Hash
Shuffle Hash
Sort-Merge
No explicit hint
```

all produced the same row count in our test.

The strategy changes **HOW** Spark executes the join, not the logical relationship being requested.

Physical strategy also does not guarantee a particular output row order.

---

## 4. Observed in Our Workspace

Environment:

```text
Spark      : 4.2.0
Compute    : Databricks Serverless
Execution  : Photon
```

Relationship:

```text
orders × customer

orders rows   = 7,500,000
customer rows =   750,000
```

Observed initial `explain()` plans:

```text
No hint
requested : optimizer decides
plan      : PhotonShuffledHashJoin
rows      : 7,500,000


MERGE
requested : SortMergeJoin
plan      : SortMergeJoin
rows      : 7,500,000


SHUFFLE_HASH
requested : ShuffledHashJoin
plan      : PhotonShuffledHashJoin
rows      : 7,500,000


BROADCAST
requested : BroadcastHashJoin
plan      : PhotonBroadcastHashJoin
rows      : 7,500,000
```

All four versions returned:

```text
7,500,000 rows
```

Therefore:

```text
all row counts equal = True
```

---

## 5. What the Hints Demonstrated

For this join, each requested strategy was reflected in the initial `explain()` plan.

### No Hint

Spark selected:

```text
PhotonShuffledHashJoin
```

### MERGE Hint

The plan changed to:

```text
SortMergeJoin
```

The operator appeared without a Photon prefix in the inspected plan.

This alone does not prove whether the entire operation executed outside Photon.

### SHUFFLE_HASH Hint

The plan showed:

```text
PhotonShuffledHashJoin
```

### BROADCAST Hint

The plan changed from:

```text
PhotonShuffledHashJoin
```

to:

```text
PhotonBroadcastHashJoin
```

for the same logical `orders × customer` relationship.

Therefore the explicit broadcast hint changed the optimizer's planned physical strategy in this example.

---

## 6. Automatic Broadcast Observation

Without a broadcast hint:

```text
orders × customer
→ PhotonShuffledHashJoin
```

With:

```python
customer.hint("broadcast")
```

the initial plan became:

```text
PhotonBroadcastHashJoin
```

Therefore Spark did **not automatically choose broadcast** for this join in the inspected plan.

We cannot prove exactly why.

Possible optimizer inputs include:

```text
available size estimates
statistics
configuration
join characteristics
AQE
```

The exact cause was not established by our experiment.

Do not conclude:

```text
customer was definitely too large
```

because the workspace did not expose enough evidence to prove that specific reason.

---

## 7. Statistics Observation

Our inspection indicated that useful table statistics were not available for both source relations.

When accurate statistics are missing, the optimizer must rely more heavily on estimates when choosing plans.

This makes statistics important for decisions such as:

```text
join strategy
build side
join order
```

Detailed statistics collection and optimizer tuning are outside the current S4.4 scope.

---

## 8. AQE Caution

The inspected plans showed:

```text
isFinalPlan=false
```

Therefore they were initial/current AQE plans rather than confirmed final runtime plans.

AQE can re-optimize a physical plan while the query executes.

So our precise conclusion is:

```text
The initial explain plans matched the requested strategies.
```

We should not claim:

```text
The final runtime strategy was definitely identical
```

unless it is confirmed through the final execution plan or Query Profile.

---

## 9. Not Verified

We did not verify:

- which strategy was fastest
- execution time differences
- final AQE operator after execution for every hinted query
- pure non-equi join strategy
- cross-join strategy
- whether the MERGE operation executed partly outside Photon
- whether a broadcast hint works when automatic broadcast is genuinely disabled
- the actual automatic broadcast threshold in this Serverless workspace
- the default join strategy of open-source Spark for large equi-joins

Those should not be recorded as observed facts.

---

## 10. Checkpoint

### Q1. What are the three main strategies covered here?

```text
Broadcast Hash Join
Shuffle Hash Join
Sort-Merge Join
```

---

### Q2. What does a BROADCAST hint request?

It requests that the specified relation be used as the broadcast side when the strategy is supported.

---

### Q3. What does SHUFFLE_HASH request?

Both sides are partitioned by the join key and Spark performs hash-based matching within corresponding partitions.

---

### Q4. What does MERGE request?

A sort-merge strategy:

```text
shuffle
→ sort
→ merge matching keys
```

---

### Q5. Does changing the strategy change the logical join result?

It should not.

In our experiment every strategy returned:

```text
7,500,000 rows
```

---

### Q6. Which strategy did Spark select without a hint?

The inspected plan showed:

```text
PhotonShuffledHashJoin
```

---

### Q7. What happened when we added a BROADCAST hint?

The initial plan changed to:

```text
PhotonBroadcastHashJoin
```

while the row count remained:

```text
7,500,000
```

---

### Q8. Did all requested hints appear in our inspected plans?

Yes.

Observed initial plans:

```text
MERGE
→ SortMergeJoin

SHUFFLE_HASH
→ PhotonShuffledHashJoin

BROADCAST
→ PhotonBroadcastHashJoin
```

However, because the plans showed:

```text
isFinalPlan=false
```

this proves the initial planned strategies, not necessarily the final AQE runtime strategy.

---

### Q9. How do we verify the actual runtime strategy?

Use the executed/final plan or Databricks Query Profile.

Do not rely only on assumptions or the hint name.

---

## 11. Core Understanding

```text
Small side available?
      ↓
Broadcast Hash may avoid large shuffle


Both sides distributed?
      ↓
Shuffle Hash may partition both sides and hash locally


Merge requested / appropriate?
      ↓
Sort-Merge shuffles + sorts + merges
```

And:

```text
Logical join
=
same business relationship

Physical strategy
=
different execution method
```

### Architect Rule

```text
Do not choose a join strategy only because
one strategy sounds faster.

Inspect:

data size
statistics
shuffle cost
memory
join type
AQE
actual execution plan
```

**S4.4 — Join Strategies COMPLETED.**